# 18. Statistical Significance: Confidence Intervals and Paired Tests for This Project's AUROC Claims

An external critique of `STUDY_PLAN.md` (treated as a paper draft) raised a real methodological gap:
every AUROC/AURC number reported elsewhere in this project (the disagreement matrix, the OOD suite,
the LLM extension) is a bare point estimate, with no confidence interval or significance test attached.
A reviewer could reasonably dismiss any comparison here as noise, since nothing so far rules that out.

`src/deployment_reliability/significance.py` (new this notebook) adds two tools:

- **`bootstrap_auroc_ci`** — a generic nonparametric bootstrap CI for a single AUROC (resample the
  positive/negative groups independently, recompute AUROC per draw, report the percentile CI).
- **`delong_test`** — the DeLong (1988) paired test for two *correlated* AUROCs computed from two
  different scores on the *same* test-set examples (e.g. combiner S vs. raw MSP, both evaluated on the
  same `id_test` correctness labels) — implemented via the structural-components/Mann-Whitney-U
  reformulation (Sun & Xu, 2014), verified in `tests/test_significance.py` against an independent
  from-scratch paired permutation test and a null-calibration simulation before being trusted here.

This notebook applies both to the four comparisons the critique specifically asked for, using only
already-cached data (`data/*.pt`) — no new downloads, no new model inference, and no fitting on any
test-only split (`id_test`, `imagenet_a`, `imagenet_o`, the OOD suite are evaluated on only, exactly as
`DESIGN.md` §10.5 already requires elsewhere in this project).

In [1]:
import os
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
torch.manual_seed(0)

from deployment_reliability.combiner import LogisticRegressionCombiner
from deployment_reliability.features import featurize, msp
from deployment_reliability.router import auroc
from deployment_reliability.significance import bootstrap_auroc_ci, delong_test

DATA_DIR = os.path.join("..", "data")


def mask(splits_arr, name):
    return torch.from_numpy(np.array(splits_arr) == name)


## Comparison 1 — Combiner AUROC vs. MSP-alone AUROC on `id_test`, ResNet-50

This is the project's core positive result: the fitted `LogisticRegressionCombiner` score `S` should
discriminate correct from incorrect predictions on `id_test` better than raw MSP alone. Both scores are
evaluated on the *same* `id_test` examples against the *same* correctness labels — the textbook DeLong
scenario (two correlated ROC curves, same underlying samples), not two independent samples.

In [2]:
cache = torch.load(os.path.join(DATA_DIR, "logit_cache_resnet50.pt"))
logits, labels = cache["logits"], cache["labels"]
splits_arr = cache["splits"]

m_fit = mask(splits_arr, "combiner_fit")
m_test = mask(splits_arr, "id_test")
correct = logits.argmax(dim=-1) == labels
phi = featurize(logits)

combiner = LogisticRegressionCombiner().fit(phi[m_fit], correct[m_fit].float())
s_combiner_test = combiner.score(phi[m_test])
s_msp_test = msp(logits[m_test])
correct_test = correct[m_test]

print(f"id_test n={int(m_test.sum())}, combiner_fit n={int(m_fit.sum())}")
print(f"combiner AUROC(corr_id) = {auroc(s_combiner_test[correct_test], s_combiner_test[~correct_test]):.4f}")
print(f"MSP-alone AUROC(corr_id) = {auroc(s_msp_test[correct_test], s_msp_test[~correct_test]):.4f}")


id_test n=1500, combiner_fit n=1500
combiner AUROC(corr_id) = 0.8841
MSP-alone AUROC(corr_id) = 0.8279


In [3]:
result_1 = delong_test(correct_test, s_combiner_test, s_msp_test)
print(f"DeLong test: AUC(combiner)={result_1.auc_a:.4f}  AUC(MSP)={result_1.auc_b:.4f}")
print(f"difference = {result_1.auc_diff:+.4f}   z = {result_1.z:.3f}   p = {result_1.p_value:.2e}")

ci_combiner_1 = bootstrap_auroc_ci(s_combiner_test[correct_test], s_combiner_test[~correct_test], n_bootstrap=3000, seed=0)
ci_msp_1 = bootstrap_auroc_ci(s_msp_test[correct_test], s_msp_test[~correct_test], n_bootstrap=3000, seed=0)
print(f"\nbootstrap 95% CI, combiner: [{ci_combiner_1.ci_lo:.4f}, {ci_combiner_1.ci_hi:.4f}]")
print(f"bootstrap 95% CI, MSP:      [{ci_msp_1.ci_lo:.4f}, {ci_msp_1.ci_hi:.4f}]")

if result_1.p_value < 0.001:
    verdict_1 = "p < 0.001 - the combiner's improvement over MSP is real, not sampling noise."
elif result_1.p_value < 0.05:
    verdict_1 = f"p = {result_1.p_value:.4f} - significant at the conventional 0.05 level."
else:
    verdict_1 = f"p = {result_1.p_value:.4f} - NOT significant at the conventional 0.05 level."
print(f"\nVERDICT: {verdict_1}")


DeLong test: AUC(combiner)=0.8841  AUC(MSP)=0.8279
difference = +0.0562   z = 6.297   p = 3.04e-10



bootstrap 95% CI, combiner: [0.8664, 0.9005]
bootstrap 95% CI, MSP:      [0.8029, 0.8525]

VERDICT: p < 0.001 - the combiner's improvement over MSP is real, not sampling noise.


## Comparison 2 — Cross-architecture: is ResNet-50 vs. ViT-B/16 vs. ConvNeXt-Tiny's `id_test`
combiner AUROC difference statistically meaningful?

These three `id_test` splits come from three separately-collected caches with different sizes
(ResNet-50: 1,500; ViT-B/16 and ConvNeXt-Tiny: 300 each) and are not the same paired samples — this is
**not** a DeLong scenario. The right tool is an independent bootstrap CI per backbone, then checking
whether the intervals overlap.

In [4]:
backbone_files = {
    "ResNet-50": "logit_cache_resnet50.pt",
    "ViT-B/16": "logit_cache_vit_b16.pt",
    "ConvNeXt-Tiny": "logit_cache_convnext_tiny.pt",
}

backbone_results = {}
for name, fname in backbone_files.items():
    c = torch.load(os.path.join(DATA_DIR, fname))
    lg, lb = c["logits"], c["labels"]
    sp = c["splits"]
    m_f, m_t = mask(sp, "combiner_fit"), mask(sp, "id_test")
    corr = lg.argmax(dim=-1) == lb
    ph = featurize(lg)
    comb = LogisticRegressionCombiner().fit(ph[m_f], corr[m_f].float())
    s_t = comb.score(ph[m_t])
    corr_t = corr[m_t]
    ci = bootstrap_auroc_ci(s_t[corr_t], s_t[~corr_t], n_bootstrap=3000, seed=0)
    backbone_results[name] = ci
    print(f"{name:16s} n(id_test)={int(m_t.sum()):5d}  AUROC={ci.auroc:.4f}  95% CI=[{ci.ci_lo:.4f}, {ci.ci_hi:.4f}]")


ResNet-50        n(id_test)= 1500  AUROC=0.8841  95% CI=[0.8664, 0.9005]


ViT-B/16         n(id_test)=  300  AUROC=0.8852  95% CI=[0.8449, 0.9213]


ConvNeXt-Tiny    n(id_test)=  300  AUROC=0.8899  95% CI=[0.8448, 0.9294]


In [5]:
names = list(backbone_results.keys())
print("Pairwise CI-overlap check (conservative - non-overlap implies significance, overlap is inconclusive not proof of no difference):")
any_overlap = False
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = backbone_results[names[i]], backbone_results[names[j]]
        overlap = not (a.ci_hi < b.ci_lo or b.ci_hi < a.ci_lo)
        any_overlap = any_overlap or overlap
        tag = "OVERLAP (not distinguishable at 95%)" if overlap else "NO OVERLAP (distinguishable at 95%)"
        print(f"  {names[i]:14s} vs {names[j]:14s}: {tag}")

print()
if any_overlap:
    print("VERDICT: at least one pair of backbones has overlapping 95% CIs on id_test combiner AUROC -")
    print("the numeric ranking (ResNet-50/ViT-B16/ConvNeXt-Tiny all near 0.88-0.89) is NOT cleanly")
    print("statistically separable from noise at this sample size, for every pair.")
else:
    print("VERDICT: all three backbones' id_test combiner AUROCs are pairwise distinguishable at 95% confidence.")


Pairwise CI-overlap check (conservative - non-overlap implies significance, overlap is inconclusive not proof of no difference):
  ResNet-50      vs ViT-B/16      : OVERLAP (not distinguishable at 95%)
  ResNet-50      vs ConvNeXt-Tiny : OVERLAP (not distinguishable at 95%)
  ViT-B/16       vs ConvNeXt-Tiny : OVERLAP (not distinguishable at 95%)

VERDICT: at least one pair of backbones has overlapping 95% CIs on id_test combiner AUROC -
the numeric ranking (ResNet-50/ViT-B16/ConvNeXt-Tiny all near 0.88-0.89) is NOT cleanly
statistically separable from noise at this sample size, for every pair.


## Comparison 3 — Real OOD suite (Places365/DTD/iNaturalist) vs. the ImageNet-O baseline (0.567)

`DESIGN.md` §24.4 reports combiner AUROC(id vs. OOD) of 0.8026/0.7949/0.7553 for Places365/DTD/iNaturalist
respectively (full-scale ImageNet-1k `id`, n=50,000, against the same already-frozen combiner fit on the
small-scale `combiner_fit` split), "clearly above" the small-scale ImageNet-O AUROC of 0.567 (`id_test`,
n=1,500). These two `id` reference sets differ (full-scale vs. small-scale), so this is again not a
DeLong scenario — bootstrap CI per dataset, then check whether the OOD-suite CIs sit clear of ImageNet-O's.

In [6]:
small_cache = torch.load(os.path.join(DATA_DIR, "logit_cache_resnet50.pt"))
small_logits, small_labels = small_cache["logits"], small_cache["labels"]
small_splits = small_cache["splits"]
m_fit_small = mask(small_splits, "combiner_fit")
m_test_small = mask(small_splits, "id_test")
m_o_small = mask(small_splits, "imagenet_o")
correct_small = small_logits.argmax(dim=-1) == small_labels
phi_small = featurize(small_logits)

combiner_ood = LogisticRegressionCombiner().fit(phi_small[m_fit_small], correct_small[m_fit_small].float())
s_id_test_small = combiner_ood.score(phi_small[m_test_small])
s_o = combiner_ood.score(phi_small[m_o_small])

ci_o_baseline = bootstrap_auroc_ci(s_id_test_small, s_o, n_bootstrap=3000, seed=0)
print(f"ImageNet-O baseline: AUROC={ci_o_baseline.auroc:.4f}  95% CI=[{ci_o_baseline.ci_lo:.4f}, {ci_o_baseline.ci_hi:.4f}]  (n_id={int(m_test_small.sum())}, n_o={int(m_o_small.sum())})")


ImageNet-O baseline: AUROC=0.5673  95% CI=[0.5475, 0.5876]  (n_id=1500, n_o=2000)


In [7]:
full_cache_path = os.path.join(DATA_DIR, "logit_cache_imagenet1k_resnet50.pt")
assert os.path.exists(full_cache_path), f"{full_cache_path} not found"
full_cache = torch.load(full_cache_path)
full_logits = full_cache["logits"]
phi_full = featurize(full_logits)
s_id_full = combiner_ood.score(phi_full)  # SAME already-frozen combiner, evaluate-only (DESIGN.md 10.5)
print(f"full-scale id n={full_logits.shape[0]}")

ood_ci_results = {}
for name in ("places365", "dtd", "inaturalist"):
    cpath = os.path.join(DATA_DIR, f"logit_cache_{name}_resnet50.pt")
    assert os.path.exists(cpath), f"{cpath} not found"
    oc = torch.load(cpath)
    ood_logits = oc["logits"]
    phi_ood = featurize(ood_logits)
    s_ood = combiner_ood.score(phi_ood)
    ci = bootstrap_auroc_ci(s_id_full, s_ood, n_bootstrap=3000, seed=0)
    ood_ci_results[name] = ci
    print(f"{name:12s} n={ood_logits.shape[0]:5d}  AUROC={ci.auroc:.4f}  95% CI=[{ci.ci_lo:.4f}, {ci.ci_hi:.4f}]")


full-scale id n=50000


places365    n= 3000  AUROC=0.8026  95% CI=[0.7954, 0.8100]


dtd          n= 5640  AUROC=0.7949  95% CI=[0.7889, 0.8009]


inaturalist  n= 3000  AUROC=0.7553  95% CI=[0.7467, 0.7638]


In [8]:
print(f"ImageNet-O baseline 95% CI: [{ci_o_baseline.ci_lo:.4f}, {ci_o_baseline.ci_hi:.4f}]\n")
for name, ci in ood_ci_results.items():
    clear = ci.ci_lo > ci_o_baseline.ci_hi
    print(f"{name:12s} 95% CI=[{ci.ci_lo:.4f}, {ci.ci_hi:.4f}]  clear of ImageNet-O's CI: {clear}")

all_clear = all(ci.ci_lo > ci_o_baseline.ci_hi for ci in ood_ci_results.values())
print()
if all_clear:
    print("VERDICT: every real OOD dataset's 95% CI sits entirely above the ImageNet-O baseline's 95% CI -")
    print("'clearly above' is statistically real, not just numerically larger, for all three datasets.")
else:
    print("VERDICT: at least one real OOD dataset's CI overlaps the ImageNet-O baseline's CI - the")
    print("'clearly above' framing needs qualifying for that dataset specifically.")


ImageNet-O baseline 95% CI: [0.5475, 0.5876]

places365    95% CI=[0.7954, 0.8100]  clear of ImageNet-O's CI: True
dtd          95% CI=[0.7889, 0.8009]  clear of ImageNet-O's CI: True
inaturalist  95% CI=[0.7467, 0.7638]  clear of ImageNet-O's CI: True

VERDICT: every real OOD dataset's 95% CI sits entirely above the ImageNet-O baseline's 95% CI -
'clearly above' is statistically real, not just numerically larger, for all three datasets.


## Comparison 4 — GPT-2 vs. Pythia-160m token-level AUROC (0.83 vs. 0.81)

`DESIGN.md` §29.4 reports token-level correctness AUROC of 0.83 (GPT-2) vs. 0.8134 (Pythia-160m) as a
close replication. The external critique used this exact pair as its own worked example of an untested
numeric claim. Each model's `id_test` is model-specific (correctness is defined per-model), so again not
a DeLong pairing — bootstrap CI per model, then compare.

In [9]:
llm_results = {}
for name, fname in [("GPT-2", "llm_feature_cache_gpt2.pt"), ("Pythia-160m", "llm_feature_cache_pythia160m.pt")]:
    c = torch.load(os.path.join(DATA_DIR, fname))
    phi_llm, correct_llm, splits_llm = c["phi"], c["correct"], c["splits"]
    m_f = mask(splits_llm, "combiner_fit")
    m_t = mask(splits_llm, "id_test")
    comb = LogisticRegressionCombiner().fit(phi_llm[m_f], correct_llm[m_f].float())
    s_t = comb.score(phi_llm[m_t])
    corr_t = correct_llm[m_t]
    ci = bootstrap_auroc_ci(s_t[corr_t], s_t[~corr_t], n_bootstrap=3000, seed=0)
    llm_results[name] = ci
    print(f"{name:12s} n(id_test)={int(m_t.sum()):7d}  AUROC={ci.auroc:.4f}  95% CI=[{ci.ci_lo:.4f}, {ci.ci_hi:.4f}]")


GPT-2        n(id_test)= 128898  AUROC=0.8316  95% CI=[0.8294, 0.8340]


Pythia-160m  n(id_test)= 128898  AUROC=0.8134  95% CI=[0.8109, 0.8158]


In [10]:
gpt2_ci, pythia_ci = llm_results["GPT-2"], llm_results["Pythia-160m"]
overlap = not (gpt2_ci.ci_hi < pythia_ci.ci_lo or pythia_ci.ci_hi < gpt2_ci.ci_lo)
print(f"GPT-2 95% CI:       [{gpt2_ci.ci_lo:.4f}, {gpt2_ci.ci_hi:.4f}]")
print(f"Pythia-160m 95% CI: [{pythia_ci.ci_lo:.4f}, {pythia_ci.ci_hi:.4f}]")
print(f"\nCIs overlap: {overlap}")
if overlap:
    print("VERDICT: GPT-2's and Pythia-160m's token-level AUROC 95% CIs overlap - the 0.83 vs. 0.81 gap is")
    print("NOT statistically distinguishable at this sample size; the two models replicate each other's")
    print("discrimination quality within noise, which is if anything a STRONGER replication result than a")
    print("numerically-close-but-'real'-difference would have been.")
else:
    print("VERDICT: the 0.83 vs. 0.81 gap is statistically real at 95% confidence, not just numerically close.")


GPT-2 95% CI:       [0.8294, 0.8340]
Pythia-160m 95% CI: [0.8109, 0.8158]

CIs overlap: False
VERDICT: the 0.83 vs. 0.81 gap is statistically real at 95% confidence, not just numerically close.


## Summary

All four numbers requested by the external critique, now with real confidence intervals / p-values
computed on this project's actual cached data (not estimated, not asserted) - see the printed VERDICT
line after each comparison above for the precise, checked conclusion. `STUDY_PLAN.md` and
`DESIGN.md` are updated with these results next to each metric's original report, cross-checked against
this notebook's actual printed output before being typed into prose.